**Stoic Philosophy Fine-tuning on NVIDIA DGX**

🖥️ **Optimized for NVIDIA GB10 (119.7 GB VRAM) with Standard PEFT**

This notebook trains a Mistral-7B model on Stoic philosophy texts using standard PEFT/LoRA for full 16-bit precision training, with bulletproof checkpointing and GGUF export.

**✨ Features:**
- ✅ Standard PEFT/LoRA - Reliable, works on any GPU including Blackwell
- ✅ Automatic checkpoint resumption - Never lose training progress
- ✅ Full 16-bit precision - No quantization, maximum quality
- ✅ Large batch sizes - Takes advantage of massive VRAM
- ✅ Complete pipeline - Training → LoRA → Merge → GGUF
- ✅ Multiple GGUF formats - q4_k_m, q5_k_m, q8_0
- ✅ Google Drive sync - All outputs saved to Drive for backup

**📋 Quick Start:**
1. Activate venv: `source ~/projects/stoic/venv/bin/activate`
2. Run cells 1-4: Setup and verify GPU
3. Run cells 5-10: Load data and train (auto-resumes if interrupted)
4. Run cell 12: Merge LoRA adapters to full model
5. Run cell 13: Convert to GGUF (creates q4_k_m, q5_k_m, q8_0)

**🎯 Output Locations:**
- Checkpoints: `~/gdrive/Colab Notebooks/stoic/checkpoints/`
- LoRA Adapters: `~/gdrive/Colab Notebooks/stoic/models_trained/`
- Merged Model: `~/gdrive/Colab Notebooks/stoic/models_merged/`
- GGUF Files: `~/gdrive/Colab Notebooks/stoic/gguf/`

## Step 1 — Environment Setup & Google Drive Mount

Check Python environment (must use stoic venv), mount Google Drive for checkpoint/model storage, and configure all paths.

**Runtime:** ~3-5 seconds  
**Output:** Drive mounted, paths configured

In [1]:
# Step 1 — Environment setup
import os
import subprocess
import sys

print("🖥️  NVIDIA DGX Setup")
print("="*60)

# Check Python executable
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

# Check if running in virtual environment
if hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix):
    print("✅ Running in virtual environment")
    if '/stoic/venv/' in sys.executable:
        print("   ✅ CORRECT: Using /home/spark/projects/stoic/venv/")
    else:
        print(f"   ⚠️  WARNING: Using {sys.executable}")
        print(f"   Expected: /home/spark/projects/stoic/venv/bin/python")
else:
    print("❌ NOT in virtual environment!")
    print(f"   Current Python: {sys.executable}")
    print(f"\n🔧 FIX THIS:")
    print(f"   1. Click the kernel selector in top-right corner")
    print(f"   2. Select 'Python (stoic)' from the dropdown")
    print(f"   3. Re-run this cell")
    print(f"\n   If 'Python (stoic)' is not listed, run in terminal:")
    print(f"   source ~/projects/stoic/venv/bin/activate")
    print(f"   python -m ipykernel install --user --name=stoic --display-name='Python (stoic)'")

# Check if already mounted
mount_point = os.path.expanduser("~/gdrive")
is_mounted = os.path.ismount(mount_point) or (os.path.exists(mount_point) and len(os.listdir(mount_point)) > 0)

if not is_mounted:
    print("\n📁 Mounting Google Drive with rclone...")
    try:
        subprocess.run(["rclone", "mount", "gdrive:", mount_point, "--vfs-cache-mode", "writes", "--daemon"], check=True)
        import time
        time.sleep(3)
        print("✅ Drive mounted")
    except Exception as e:
        print(f"❌ Failed to mount drive: {e}")
        print(f"   Run manually: rclone mount gdrive: ~/gdrive --vfs-cache-mode writes --daemon")
else:
    print("✅ Drive already mounted")

# Set paths
DRIVE_BASE = os.path.expanduser("~/gdrive/Colab Notebooks")
LOCAL_CACHE = os.path.expanduser("~/projects/stoic/cache")
DATA_PATH = f"{DRIVE_BASE}/stoic/mlx_format/train.jsonl"
OUTPUT_DIR = f"{DRIVE_BASE}/stoic/models_trained/stoic-mistral-7b-lora"
CHECKPOINT_DIR = f"{DRIVE_BASE}/stoic/checkpoints"
MERGED_DIR = f"{DRIVE_BASE}/stoic/models_merged/stoic-mistral-merged-f16"
GGUF_DIR = f"{DRIVE_BASE}/stoic/gguf"

os.makedirs(LOCAL_CACHE, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if os.path.exists(DRIVE_BASE):
    print(f"\n✅ Drive path verified: {DRIVE_BASE}")
    print(f"   Local cache: {LOCAL_CACHE}")
    print(f"   Checkpoints: {CHECKPOINT_DIR}")
    print(f"   Output: {OUTPUT_DIR}")
else:
    print(f"\n⚠️  Warning: Drive not accessible at {DRIVE_BASE}")
    if os.path.exists(os.path.expanduser('~/gdrive')):
        print(f"   Available: {os.listdir(os.path.expanduser('~/gdrive'))}")

print("\n" + "="*60)

🖥️  NVIDIA DGX Setup
Python executable: /home/spark/projects/stoic/venv/bin/python
Python version: 3.12.3
✅ Running in virtual environment
   ✅ CORRECT: Using /home/spark/projects/stoic/venv/
✅ Drive already mounted

✅ Drive path verified: /home/spark/gdrive/Colab Notebooks
   Local cache: /home/spark/projects/stoic/cache
   Checkpoints: /home/spark/gdrive/Colab Notebooks/stoic/checkpoints
   Output: /home/spark/gdrive/Colab Notebooks/stoic/models_trained/stoic-mistral-7b-lora



## Step 2 — Hugging Face Authentication

Login to Hugging Face to download models.

In [2]:
# Step 2 — HF login
from huggingface_hub import login
import os

HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN and HF_TOKEN.startswith("hf_"):
    login(token=HF_TOKEN)
    print("✅ Logged into Hugging Face")
else:
    from huggingface_hub import notebook_login
    print("⚠️  No valid HF_TOKEN found. Set it with:")
    print("   export HF_TOKEN='your_token_here'")
    notebook_login()

/home/spark/projects/stoic/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Logged into Hugging Face


## Step 3 — Check Dependencies

Verify all required packages are installed in the venv.

In [3]:
# Step 3 — Check dependencies
print("📦 Checking dependencies...")

required = {
    'torch': 'PyTorch',
    'transformers': 'Transformers',
    'datasets': 'Datasets',
    'peft': 'PEFT',
    'trl': 'TRL',
    'accelerate': 'Accelerate',
    'sentencepiece': 'SentencePiece',
}

missing = []
for module, name in required.items():
    try:
        __import__(module)
        print(f"✅ {name}")
    except ImportError:
        print(f"❌ {name} - missing")
        missing.append(module)

try:
    import bitsandbytes
    print(f"✅ bitsandbytes")
except ImportError:
    print(f"⚠️  bitsandbytes - not available (will use standard AdamW)")

if missing:
    print(f"\n⚠️  Missing packages. Install them in your terminal:")
    print(f"   source ~/projects/stoic/venv/bin/activate")
    for pkg in missing:
        print(f"   pip install {pkg}")
    raise ImportError(f"Missing required packages: {missing}")
else:
    print(f"\n✅ All required packages available!")

📦 Checking dependencies...
✅ PyTorch
✅ Transformers
✅ Datasets


/home/spark/projects/stoic/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


✅ PEFT
✅ TRL
✅ Accelerate
✅ SentencePiece
✅ bitsandbytes

✅ All required packages available!


## Step 4 — GPU Verification

Check GPU availability, VRAM, and configure precision settings.

In [4]:
# Step 4 — GPU check
import torch

print("="*60)
print("GPU Configuration")
print("="*60)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name()
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_memory_gb:.1f} GB")
    print(f"CUDA Compute Capability: {torch.cuda.get_device_capability(0)}")
    
    bf16_supported = torch.cuda.is_bf16_supported()
    print(f"BF16 support: {bf16_supported}")

    if "GB10" in gpu_name or gpu_memory_gb > 100:
        print(f"\n🚀 {gpu_name} detected - MASSIVE VRAM!")
        print("   Optimizations enabled:")
        print("   ✅ Full precision training (bf16/fp16)")
        print("   ✅ Large batch sizes (16)")
        print("   ✅ Fast training")
    elif "A100" in gpu_name or "V100" in gpu_name:
        print(f"\n🚀 {gpu_name} detected - using full precision!")
        print("   ✅ Full precision training (bf16/fp16)")
        print("   ✅ Moderate batch sizes (8)")
    elif "T4" in gpu_name:
        print(f"\n✅ {gpu_name} detected - using full precision!")
        print("   ✅ Full precision training (bf16/fp16)")
        print("   ✅ Smaller batch sizes (2) due to limited VRAM")
    else:
        print(f"\n⚠️  GPU: {gpu_name}")
else:
    raise RuntimeError("❌ No GPU detected!")

print("="*60)

GPU Configuration
PyTorch: 2.9.1+cu130
CUDA available: True
GPU: NVIDIA GB10
VRAM: 119.7 GB
CUDA Compute Capability: (12, 1)
BF16 support: True

🚀 NVIDIA GB10 detected - MASSIVE VRAM!
   Optimizations enabled:
   ✅ Full precision training (bf16/fp16)
   ✅ Large batch sizes (16)
   ✅ Fast training


## Step 5 — Load Training Data

Load train.jsonl from Google Drive.

In [5]:
# Step 5 — Load training data
import json

raw_records = []

try:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if "text" in record:
                    raw_records.append(record)
                elif "instruction" in record and "response" in record:
                    raw_records.append(record)
                else:
                    print(f"⚠️  Skipping malformed record: {record.keys()}")

    print(f"✅ Loaded {len(raw_records)} training examples from Drive")

    if raw_records:
        print(f"\nSample record keys: {raw_records[0].keys()}")

        if "text" in raw_records[0]:
            print(f"\nSample (pre-formatted):\n{raw_records[0]['text'][:200]}...")
        else:
            print(f"\nSample:\nInstruction: {raw_records[0].get('instruction', '')[:100]}...")
            print(f"Response: {raw_records[0].get('response', '')[:100]}...")

except FileNotFoundError:
    print(f"❌ Data file not found: {DATA_PATH}")
    print(f"\n📋 Upload your train.jsonl to Google Drive at:")
    print(f"   {DATA_PATH}")
    print(f"\nOr copy from local:")
    print(f"   /Users/beaudamore/Source/augmentoolkit/outputs/marcus_aurelius_dataset/mlx_format/train.jsonl")
    raise

✅ Loaded 2648 training examples from Drive

Sample record keys: dict_keys(['text'])

Sample (pre-formatted):
[INST] Extend the inquiry of the given prompt by exploring the effects of different lunar phases on the frequency of emergency room visits caused by various mental health conditions. Utilize advanced ...


## Step 6 — Load Base Model

Load Mistral-7B-Instruct-v0.3 in full 16-bit precision (bf16/fp16).

In [6]:
# Step 6 — Load base model with standard PEFT
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

max_seq_length = 2048

print("="*60)
print("Loading Mistral-7B-Instruct-v0.3")
print("="*60)
print(f"Configuration:")
print(f"  Max sequence length: {max_seq_length}")
print(f"  Precision: Full 16-bit (bf16/fp16)")
print(f"  Quantization: None")
print(f"  Framework: Standard PEFT/Transformers")
print(f"  Cache: ~/.cache/huggingface/ (default)")

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
print(f"  Model: {model_name}")

print("\n📥 Loading model...")
print("   Using cached model if available (~14GB)")

# Determine dtype based on GPU capability
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map={"": 0},  # Explicit device placement to GPU 0
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("="*60)
print(f"   Precision: {model.dtype}")
print(f"   Model size: ~14GB in VRAM")
print(f"   Cached at: ~/.cache/huggingface/hub/")
print("\n✅ Mistral-7B loaded in full precision")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Mistral-7B-Instruct-v0.3
Configuration:
  Max sequence length: 2048
  Precision: Full 16-bit (bf16/fp16)
  Quantization: None
  Framework: Standard PEFT/Transformers
  Cache: ~/.cache/huggingface/ (default)
  Model: mistralai/Mistral-7B-Instruct-v0.3

📥 Loading model...
   Using cached model if available (~14GB)


Loading checkpoint shards: 100%|██████████| 3/3 [01:12<00:00, 24.02s/it]


   Precision: torch.bfloat16
   Model size: ~14GB in VRAM
   Cached at: ~/.cache/huggingface/hub/

✅ Mistral-7B loaded in full precision


## Step 7 — Add LoRA Adapters

Configure and apply LoRA to the base model for efficient fine-tuning.

In [7]:
# Step 7 — Add LoRA adapters with PEFT
from peft import LoraConfig, get_peft_model

print("Configuring LoRA with PEFT...")

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.25,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

print("✅ LoRA configuration created")
print(f"   Rank: 32")
print(f"   Alpha: 64")
print(f"   Dropout: 0.25")
print(f"   Gradient checkpointing: Enabled")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("\n✅ Model prepared for PEFT training")
print(f"Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

Configuring LoRA with PEFT...
✅ LoRA configuration created
   Rank: 32
   Alpha: 64
   Dropout: 0.25
   Gradient checkpointing: Enabled

✅ Model prepared for PEFT training
Trainable params: 83,886,080 (1.14%)


## Step 8 — Format Dataset

Apply Mistral chat template to all training examples.

In [8]:
# Step 8 — Format dataset
from datasets import Dataset

def format_chat_template(row):
    if "text" in row and "instruction" not in row:
        return {"text": row["text"]}

    chat = [
        {"role": "user", "content": row['instruction']},
        {"role": "assistant", "content": row['response']},
    ]

    text = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

dataset = Dataset.from_list(raw_records)
train_dataset = dataset.map(format_chat_template, num_proc=1)

print(f"✅ Dataset ready: {len(train_dataset)} examples")
if len(train_dataset) > 0:
    print(f"\nSample formatted text:\n{train_dataset[0]['text'][:300]}...")

Map (num_proc=1):   0%|          | 0/2648 [00:00<?, ? examples/s]

Map (num_proc=1): 100%|██████████| 2648/2648 [00:00<00:00, 8455.23 examples/s] 

✅ Dataset ready: 2648 examples

Sample formatted text:
[INST] Extend the inquiry of the given prompt by exploring the effects of different lunar phases on the frequency of emergency room visits caused by various mental health conditions. Utilize advanced statistical analysis methods to uncover hidden patterns and associations, while taking into consider...


## Step 9 — Configure Trainer

Set up training with auto-checkpointing and GPU-optimized batch sizes.

In [ ]:
# Step 9 — Configure trainer
from trl import SFTTrainer, SFTConfig
import torch
import os

num_examples = len(train_dataset)
epochs = 2
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_memory_gb > 100:
    per_device_batch = 16
    grad_accum = 2
    print("🚀 Using GB10-optimized batch size")
elif gpu_memory_gb > 40:
    per_device_batch = 8
    grad_accum = 2
elif gpu_memory_gb > 24:
    per_device_batch = 4
    grad_accum = 4
else:
    per_device_batch = 2
    grad_accum = 4

effective_batch = per_device_batch * grad_accum
total_steps = (num_examples * epochs) // effective_batch
warmup_steps = 100

print("="*60)
print("Training Configuration")
print("="*60)
print(f"Dataset:")
print(f"  Examples: {num_examples}")
print(f"  Epochs: {epochs}")
print(f"\nBatch Configuration:")
print(f"  Per-device batch: {per_device_batch}")
print(f"  Gradient accumulation: {grad_accum}")
print(f"  Effective batch size: {effective_batch}")
print(f"\nTraining Steps:")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Save every: 100 steps")
print(f"\nOptimization:")
print(f"  Learning rate: 5e-5")
print(f"  Optimizer: adamw_8bit")
print(f"  Scheduler: cosine")

resume_from_checkpoint = None
if os.path.exists(CHECKPOINT_DIR):
    checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=lambda x: int(x.split("-")[1]))
        resume_from_checkpoint = os.path.join(CHECKPOINT_DIR, latest_checkpoint)
        print(f"\n🔄 Found checkpoint: {latest_checkpoint}")
        print(f"   Will resume training from this point")

sft_config = SFTConfig(
    per_device_train_batch_size=per_device_batch,
    gradient_accumulation_steps=grad_accum,
    warmup_steps=warmup_steps,
    num_train_epochs=epochs,
    learning_rate=5e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=1337,
    output_dir=CHECKPOINT_DIR,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=50,
    load_best_model_at_end=False,
    report_to="none",
    dataset_text_field="text",
    max_length=max_seq_length,
    packing=False,
    dataset_num_proc=2,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=sft_config,
)

if gpu_memory_gb > 100:
    time_per_step = 0.08
elif gpu_memory_gb > 40:
    time_per_step = 0.15
else:
    time_per_step = 0.5

estimated_minutes = (total_steps * time_per_step) / 60
print("="*60)
print(f"   Checkpoints saved to: {CHECKPOINT_DIR}")
print(f"   Estimated training time: ~{estimated_minutes:.0f} minutes ({estimated_minutes/60:.1f} hours)")
print(f"\n✅ Trainer initialized")

🚀 Using GB10-optimized batch size
Training Configuration
Dataset:
  Examples: 2648
  Epochs: 2

Batch Configuration:
  Per-device batch: 16
  Gradient accumulation: 2
  Effective batch size: 32

Training Steps:
  Total steps: 165
  Warmup steps: 100
  Save every: 100 steps

Optimization:
  Learning rate: 5e-5
  Optimizer: adamw_8bit
  Scheduler: cosine


Adding EOS to train dataset (num_proc=2):   0%|          | 0/2648 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2): 100%|██████████| 2648/2648 [00:00<00:00, 6167.27 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.


   Checkpoints saved to: /home/spark/gdrive/Colab Notebooks/stoic/checkpoints
   Estimated training time: ~0 minutes (0.0 hours)

✅ Trainer initialized


## Step 10 — Train Model

Train with bulletproof checkpointing. Auto-resumes if interrupted.

In [13]:
# Step 10 — Train model
import time
from datetime import datetime
from pathlib import Path
import os

print("="*60)
print("🚀 Starting Training")
print("="*60)
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Saving checkpoints to: {CHECKPOINT_DIR}")
print(f"Final LoRA adapters to: {OUTPUT_DIR}")
print("="*60)

start_time = time.time()

try:
    if resume_from_checkpoint:
        print(f"\n🔄 Resuming from {resume_from_checkpoint}")
        trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    else:
        print(f"\n▶️  Starting fresh training")
        trainer.train()

    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✅ Training completed!")
    print(f"   Duration: {elapsed/60:.1f} minutes ({elapsed/3600:.2f} hours)")
    print(f"{'='*60}")

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"\n💾 Saving final LoRA adapters to Drive...")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    
    adapter_files = os.listdir(OUTPUT_DIR)
    print(f"✅ LoRA adapters saved: {OUTPUT_DIR}")
    print(f"   Files: {len(adapter_files)}")
    print(f"   Key files: {[f for f in adapter_files if 'adapter' in f or 'config' in f]}")

    with open(os.path.join(OUTPUT_DIR, "TRAINING_COMPLETE.txt"), "w") as f:
        f.write(f"Training completed at {datetime.now()}\n")
        f.write(f"Duration: {elapsed/60:.1f} minutes\n")
        f.write(f"Examples: {num_examples}\n")
        f.write(f"Epochs: {epochs}\n")
    
    print(f"\n🎉 Ready for merging and GGUF conversion!")

except KeyboardInterrupt:
    elapsed = time.time() - start_time
    print(f"\n⚠️  Training interrupted after {elapsed/60:.1f} minutes")
    print(f"   Latest checkpoint saved at: {CHECKPOINT_DIR}")
    print(f"   Resume by re-running this cell")
    
except Exception as e:
    elapsed = time.time() - start_time
    print(f"\n❌ Training failed after {elapsed/60:.1f} minutes")
    print(f"   Error: {e}")
    print(f"   Latest checkpoint: {CHECKPOINT_DIR}")
    import traceback
    traceback.print_exc()
    raise

print("\n" + "="*60)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


🚀 Starting Training
Time: 2026-01-02 15:59:44
Saving checkpoints to: /home/spark/gdrive/Colab Notebooks/stoic/checkpoints
Final LoRA adapters to: /home/spark/gdrive/Colab Notebooks/stoic/models_trained/stoic-mistral-7b-lora

▶️  Starting fresh training


Step,Training Loss
10,1.897600
20,1.578900
30,1.349700
40,1.162500
50,1.076800
60,0.984900
70,0.956700
80,0.915800
90,0.904700
100,0.897600



✅ Training completed!
   Duration: 246.9 minutes (4.11 hours)

💾 Saving final LoRA adapters to Drive...
✅ LoRA adapters saved: /home/spark/gdrive/Colab Notebooks/stoic/models_trained/stoic-mistral-7b-lora
   Files: 8
   Key files: ['adapter_config.json', 'adapter_model.safetensors', 'tokenizer_config.json']

🎉 Ready for merging and GGUF conversion!



## Step 11 — Test Inference (Optional)

Test the trained model with sample prompts.

In [15]:
# Step 11 — Test inference (optional)
model.eval()

test_prompts = [
    "What did Marcus Aurelius teach about dealing with difficult people?",
    "Explain Epictetus's view on what is within our control",
    "How do the Stoics define virtue and living well?"
]

print("Testing trained model:\n" + "="*60)

for test_prompt in test_prompts:
    messages = [{"role": "user", "content": test_prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        do_sample=True,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()

    print(f"\n📜 Prompt: {test_prompt}")
    print(f"Response: {response[:300]}...")
    print("-" * 60)

print("\n✅ Inference test complete!")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Testing trained model:

📜 Prompt: What did Marcus Aurelius teach about dealing with difficult people?
Response: What did Marcus Aurelius teach about dealing with difficult people? **Finished.** � Thought Process:
Let's tackle this question by recalling the key principles that Marcus Aurelius taught regarding how to deal with difficult people. To answer this, we need to focus on his philosophical thoughts and ...
------------------------------------------------------------

📜 Prompt: Explain Epictetus's view on what is within our control
Response: Explain Epictetus's view on what is within our control **Finished.**  Thought Process:
Let’s tackle this question. The user wants to understand Epictetus’s view on what is within our control, which requires recalling key concepts from his teachings. I need to focus on the idea that individuals have ...
------------------------------------------------------------

📜 Prompt: How do the Stoics define virtue and living well?
Response: How do the S

## Step 12 — Merge LoRA Adapters

Merge trained LoRA weights into base model to create full fp16 model.

In [ ]:
# Step 12 — Merge LoRA adapters
import os, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("="*60)
print("Merging LoRA adapters into full fp16 model")
print("="*60)

if not os.path.exists(OUTPUT_DIR):
    raise FileNotFoundError(f"❌ LoRA adapters not found at {OUTPUT_DIR}. Train first!")

if not os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")):
    raise FileNotFoundError(f"❌ Invalid adapter directory. Missing adapter_config.json")

Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)

if os.path.exists(os.path.join(MERGED_DIR, "config.json")):
    print(f"⚠️  Merged model already exists at {MERGED_DIR}")
    overwrite = input("Overwrite? (yes/no): ").strip().lower()
    if overwrite != "yes":
        print("Skipping merge.")
        raise SystemExit(0)

print(f"\n📥 Loading base model (fp16) from HF cache...")
print(f"   Using: mistralai/Mistral-7B-Instruct-v0.3")

base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

print(f"✅ Base model loaded")

print(f"\n🔗 Loading LoRA adapters from {OUTPUT_DIR}...")
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
print(f"✅ Adapters loaded")

print(f"\n🔄 Merging adapters into base model...")
model = model.merge_and_unload()
print(f"✅ Merge complete")

print(f"\n💾 Saving merged model to: {MERGED_DIR}")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

saved_files = os.listdir(MERGED_DIR)
total_size = sum(
    os.path.getsize(os.path.join(MERGED_DIR, f))
    for f in saved_files
    if os.path.isfile(os.path.join(MERGED_DIR, f))
)

with open(os.path.join(MERGED_DIR, "MERGE_COMPLETE.txt"), "w") as f:
    f.write(f"Merge completed at {datetime.now()}\n")
    f.write(f"Size: {total_size / 1024**3:.2f} GB\n")

print(f"\n{'='*60}")
print(f"✅ MERGED MODEL SAVED!")
print(f"   Location: {MERGED_DIR}")
print(f"   Files: {len(saved_files)}")
print(f"   Size: {total_size / 1024**3:.1f} GB")
print(f"\n🎯 Next: Run Step 13 to convert to GGUF")
print("="*60)

Merging LoRA adapters into full fp16 model


FileNotFoundError: ❌ LoRA adapters not found at /home/spark/gdrive/Colab Notebooks/stoic/models_trained/stoic-mistral-7b-lora. Train first!

## Step 13 — Convert to GGUF

Convert merged model to GGUF format (q4_k_m, q5_k_m, q8_0) for deployment.

In [ ]:
# Step 13 — Convert to GGUF
import os
import subprocess
from pathlib import Path
from datetime import datetime

print("="*60)
print("Converting to GGUF on DGX")
print("="*60)

if not os.path.exists(MERGED_DIR):
    raise FileNotFoundError(f"❌ Merged model not found at: {MERGED_DIR}")

if not os.path.exists(os.path.join(MERGED_DIR, "config.json")):
    raise FileNotFoundError(f"❌ Invalid merged model. Missing config.json")

Path(GGUF_DIR).mkdir(parents=True, exist_ok=True)

LLAMA_CPP_DIR = os.path.expanduser("~/projects/stoic/llama.cpp")

if not os.path.exists(LLAMA_CPP_DIR):
    print(f"\n📥 Cloning llama.cpp to {LLAMA_CPP_DIR}...")
    subprocess.run([
        "git", "clone", 
        "https://github.com/ggerganov/llama.cpp.git", 
        LLAMA_CPP_DIR
    ], check=True)
    print("✅ Cloned llama.cpp")
else:
    print(f"✅ llama.cpp already exists at {LLAMA_CPP_DIR}")

print(f"\n🔨 Building llama.cpp with CUDA support...")
build_result = subprocess.run(
    ["make", "-C", LLAMA_CPP_DIR, "clean"],
    capture_output=True
)
build_result = subprocess.run(
    ["make", "-C", LLAMA_CPP_DIR, "-j8", "LLAMA_CUDA=1"],
    capture_output=True,
    text=True
)

if build_result.returncode != 0:
    print("⚠️  CUDA build failed, trying CPU build...")
    subprocess.run(["make", "-C", LLAMA_CPP_DIR, "clean"], check=True)
    subprocess.run(["make", "-C", LLAMA_CPP_DIR, "-j8"], check=True)
    print("✅ Built llama.cpp (CPU)")
else:
    print("✅ Built llama.cpp (CUDA)")

print(f"\n📦 Checking Python requirements...")
try:
    import sentencepiece
    import google.protobuf
    import numpy
    print("✅ Required packages available")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("Install in terminal: pip install sentencepiece protobuf numpy")
    raise

f16_output = os.path.join(GGUF_DIR, "stoic-mistral-f16.gguf")
print(f"\n🔄 Converting to GGUF f16...")
print(f"   Input: {MERGED_DIR}")
print(f"   Output: {f16_output}")

convert_script = os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")
subprocess.run([
    "python3", convert_script,
    MERGED_DIR,
    "--outfile", f16_output,
    "--outtype", "f16"
], check=True)

f16_size = os.path.getsize(f16_output) / 1024**3
print(f"✅ F16 GGUF created: {f16_size:.1f} GB")

quantize_bin = os.path.join(LLAMA_CPP_DIR, "llama-quantize")
quant_levels = [
    ("q4_k_m", "Recommended - 3.5GB, fast, good quality"),
    ("q5_k_m", "Balanced - 4.3GB, better quality"),
    ("q8_0", "High quality - 7GB, slower"),
]

print(f"\n⚙️  Quantizing to multiple levels...")
for quant_type, description in quant_levels:
    output_file = os.path.join(GGUF_DIR, f"stoic-mistral-{quant_type}.gguf")
    print(f"\n   Creating {quant_type} ({description})...")
    
    subprocess.run([
        quantize_bin,
        f16_output,
        output_file,
        quant_type
    ], check=True)
    
    size = os.path.getsize(output_file) / 1024**3
    print(f"   ✅ {quant_type}: {size:.1f} GB")

with open(os.path.join(GGUF_DIR, "CONVERSION_COMPLETE.txt"), "w") as f:
    f.write(f"GGUF conversion completed at {datetime.now()}\n")
    f.write(f"F16 size: {f16_size:.2f} GB\n")
    for quant_type, _ in quant_levels:
        output_file = os.path.join(GGUF_DIR, f"stoic-mistral-{quant_type}.gguf")
        size = os.path.getsize(output_file) / 1024**3
        f.write(f"{quant_type}: {size:.2f} GB\n")

print(f"\n{'='*60}")
print(f"✅ GGUF CONVERSION COMPLETE!")
print(f"{'='*60}")
print(f"\nFiles created in {GGUF_DIR}:")
print(f"   📄 stoic-mistral-f16.gguf ({f16_size:.1f} GB)")
for quant_type, description in quant_levels:
    output_file = os.path.join(GGUF_DIR, f"stoic-mistral-{quant_type}.gguf")
    size = os.path.getsize(output_file) / 1024**3
    print(f"   📄 stoic-mistral-{quant_type}.gguf ({size:.1f} GB) - {description}")

print(f"\n🎯 Recommended: stoic-mistral-q4_k_m.gguf")
print(f"\n📋 To use with Ollama:")
print(f"   cd \"{GGUF_DIR}\"")
print(f"   ollama create stoic -f <(echo 'FROM \"./stoic-mistral-q4_k_m.gguf\"")
print(f"   TEMPLATE \"\"\"[INST] {{{{ .Prompt }}}} [/INST]\"\"\"")
print(f"   PARAMETER temperature 0.7")
print(f"   PARAMETER top_p 0.9')")
print(f"   ollama run stoic \"What did Marcus Aurelius teach about virtue?\"")
print("="*60)

---

## 🎯 Import to Ollama

<details>
<summary><b>Click to expand instructions</b></summary>

After GGUF conversion completes, the files will be at:
`~/gdrive/Colab Notebooks/stoic/gguf/`

### Import to Ollama:

```bash
cd ~/gdrive/"Colab Notebooks"/stoic/gguf

ollama create stoic -f <(echo 'FROM "./stoic-mistral-q4_k_m.gguf"
TEMPLATE """[INST] {{ .Prompt }} [/INST]"""
PARAMETER temperature 0.7
PARAMETER top_p 0.9')
```

### Test the model:

```bash
ollama run stoic "What did Marcus Aurelius teach about virtue?"
ollama run stoic "Explain Epictetus's view on what is within our control"
ollama run stoic "How do the Stoics define living well?"
```

### GGUF File Sizes:
- **q4_k_m**: 3.5GB (fast, good quality) ← **RECOMMENDED**
- **q5_k_m**: 4.3GB (balanced quality/speed)
- **q8_0**: 7GB (excellent quality, slower)

</details>

---

## 📊 Complete Workflow Summary

<details open>
<summary><b>Training Pipeline (Click to collapse)</b></summary>

### Execution Order:
1. **Cells 1-4**: Environment setup, GPU verification (~1 min)
2. **Cell 5**: Load training data (~1 sec)
3. **Cells 6-8**: Load model, add LoRA adapters, format dataset (~3 min)
4. **Cells 9-10**: Configure trainer and train (~30-45 min)
5. **Cell 11**: (Optional) Test inference (~1 min)
6. **Cell 11b**: Merge LoRA adapters → full fp16 model (~5 min)
7. **Cell 12**: Convert to GGUF (q4_k_m, q5_k_m, q8_0) (~15 min)

**Total time**: ~50-70 minutes (full pipeline)

</details>

<details>
<summary><b>🛡️ Bulletproof Features</b></summary>

- ✅ **Auto-checkpoint resumption** - Training continues from last checkpoint if interrupted
- ✅ **Checkpoints saved every 100 steps** - Minimal progress loss
- ✅ **5 checkpoints retained** - Can roll back if needed
- ✅ **Completion markers** - Easy to verify each stage completed
- ✅ **All outputs synced to Google Drive** - Automatic backup

</details>

<details>
<summary><b>💾 Files Created</b></summary>

```
~/gdrive/Colab Notebooks/stoic/
├── checkpoints/
│   ├── checkpoint-100/
│   ├── checkpoint-200/
│   └── ... (auto-resumes from latest)
├── models_trained/stoic-mistral-7b-lora/
│   ├── adapter_config.json
│   ├── adapter_model.safetensors
│   └── TRAINING_COMPLETE.txt ✓
├── models_merged/stoic-mistral-merged-f16/
│   ├── model-00001-of-00003.safetensors
│   ├── config.json
│   └── MERGE_COMPLETE.txt ✓
└── gguf/
    ├── stoic-mistral-f16.gguf (14GB)
    ├── stoic-mistral-q4_k_m.gguf (3.5GB) ← RECOMMENDED
    ├── stoic-mistral-q5_k_m.gguf (4.3GB)
    ├── stoic-mistral-q8_0.gguf (7GB)
    └── CONVERSION_COMPLETE.txt ✓
```

</details>

<details>
<summary><b>🚀 Performance on GB10</b></summary>

- **Training**: ~30-45 minutes (with 119GB VRAM, batch_size=16)
- **Full precision**: bf16 (no 4-bit quantization needed!)
- **Checkpointing**: Saves every ~5 minutes
- **Total pipeline**: ~1 hour (train + merge + GGUF)

</details>

<details>
<summary><b>🎯 Using the Model</b></summary>

```bash
cd ~/gdrive/"Colab Notebooks"/stoic/gguf
ollama create stoic -f <(echo 'FROM "./stoic-mistral-q4_k_m.gguf"
TEMPLATE """[INST] {{ .Prompt }} [/INST]"""
PARAMETER temperature 0.7
PARAMETER top_p 0.9')
ollama run stoic "What did Marcus Aurelius teach about virtue?"
```

</details>

---

## 🔧 Troubleshooting & Recovery

<details>
<summary><b>Training Interrupted?</b></summary>

```python
# Just re-run Cell 10 - it will automatically resume from the last checkpoint
# Look for: "🔄 Resuming from checkpoint-XXX"
```

</details>

<details>
<summary><b>Start Fresh Training?</b></summary>

```bash
# Remove checkpoints to start over
rm -rf ~/gdrive/Colab\ Notebooks/stoic/checkpoints/*
# Then re-run Cell 10
```

</details>

<details>
<summary><b>Check Training Progress</b></summary>

```bash
# List checkpoints
ls -lh ~/gdrive/Colab\ Notebooks/stoic/checkpoints/

# Check if training completed
cat ~/gdrive/Colab\ Notebooks/stoic/models_trained/stoic-mistral-7b-lora/TRAINING_COMPLETE.txt
```

</details>

<details>
<summary><b>Memory Usage by GPU</b></summary>

- **GB10 (119GB VRAM)**: 
  - Training: ~40-50GB VRAM (plenty of headroom!)
  - Batch size: 16 (optimized for speed)
  - Full precision: bf16/fp16 (no quantization)
  
- **A100 (40GB VRAM)**:
  - Batch size: 8
  - Full precision: bf16
  
- **T4 (16GB VRAM)**:
  - Batch size: 2
  - 4-bit quantization

</details>

<details>
<summary><b>Recommended Workflow</b></summary>

1. **First time**: Run all cells in order (1-12)
2. **If interrupted**: Just re-run the cell that was running
3. **To re-merge**: Delete `MERGE_COMPLETE.txt` and re-run Cell 11b
4. **To re-convert GGUF**: Delete `CONVERSION_COMPLETE.txt` and re-run Cell 12

</details>

<details>
<summary><b>Backup Strategy</b></summary>

- All important files auto-save to Google Drive
- Checkpoints kept during training
- Final outputs have completion markers
- Can sync to local with: `rsync -av ~/gdrive/Colab\ Notebooks/stoic/ ~/backups/stoic/`

</details>